In [ ]:
# 13 — Gold: handyman scorecards
# One row per (handyman, incident type) plus an overall row (incident_type = 'all'): completed / cancelled /
# active jobs, success rate (share of rated jobs rated 4–5), average rating, average resolution time and
# travel time. Full recompute from silver; the live stream updates single handymen incrementally.

In [ ]:
import os, sys, time
sys.path.insert(0, os.getcwd())          # pipeline_lib.py lives next to this notebook
import pipeline_lib as pl
from pyspark.sql import functions as F

dbutils.widgets.text("run_id", "manual")
RUN_ID = dbutils.widgets.get("run_id")
pl.ensure_tables(spark)

started = time.time()
pl.compute_performance(spark.table(pl.T_SILVER)).createOrReplaceTempView("_perf")
spark.sql(f"""CREATE OR REPLACE TABLE {pl.T_GOLD_PERFORMANCE} CLUSTER BY (handyman_user_id)
              COMMENT 'Handyman scorecards per incident type (+ overall row incident_type = all)'
              AS SELECT * FROM _perf""")

gold = spark.table(pl.T_GOLD_PERFORMANCE)
n_rows = gold.count()
n_handymen = gold.where("incident_type = 'all'").count()
expected = spark.table(pl.T_SILVER).where("handyman_user_id IS NOT NULL").select("handyman_user_id").distinct().count()
bad_rates = gold.where("success_rate < 0 OR success_rate > 1 OR avg_rating < 1 OR avg_rating > 5").count()
display(gold.where("incident_type = 'all'").orderBy(F.desc("jobs_completed")).limit(10))

pl.log_step(spark, RUN_ID, "13_gold_performance", started, rows_in=None, rows_out=n_rows,
            checks={"every_handyman_has_overall_row": n_handymen == expected,
                    "rates_in_range": bad_rates == 0})